In [ ]:
from pyspark.sql import DataFrame

from transforms.elo import build_elo

SILVER_TABLE = "stryktipset.silver.matches"
DIM_LEAGUE_TABLE = "stryktipset.gold.dim_league"  # built by 03_gold_dimensions
GOLD_TABLE = "stryktipset.gold.fact_match"  # built by 04_gold_fact_match

In [ ]:
def load_matches(table: str = SILVER_TABLE) -> DataFrame:
    """Load the match dataset written by 02_silver_transform."""
    return spark.table(table)


def scope_to_gold_leagues(matches: DataFrame, dim_league: DataFrame) -> DataFrame:
    """Keep only the England/Sweden domestic leagues, by inner-joining
    dim_league -- the same mechanism build_fact_match uses, so the league
    list stays in transforms/dimensions.py rather than living in two
    places.

    Cups and European ties are deliberately excluded. They would tie the
    divisions onto a more comparable scale (an FA Cup tie measures across
    four tiers at once), but gold's scope is domestic league football and
    the ratings follow it. Divisions are therefore linked only by
    promoted and relegated teams carrying their ratings with them.
    """
    return matches.join(
        dim_league,
        on=matches["league"] == dim_league["league_name"],
        how="inner",
    ).select(
        matches["match_id"],
        matches["match_start"],
        matches["home_team"],
        matches["away_team"],
        matches["home_goals"],
        matches["away_goals"],
        matches["result"],
    )

In [ ]:
def merge_into_fact_match(df_elo: DataFrame, table: str = GOLD_TABLE) -> None:
    """Merge elo_home/elo_away onto the existing fact_match rows, on
    match_key.

    Named column assignment, same as 05. The columns are pre-created as
    nulls by 04 because Delta resolves a named assignment before any
    schema evolution runs -- without the placeholders this fails with
    DELTA_MERGE_UNRESOLVED_EXPRESSION.
    """
    source_view = "elo_source"
    df_elo.select("match_id", "elo_home", "elo_away").createOrReplaceTempView(source_view)

    spark.sql(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {table} AS target
        USING {source_view} AS source
        ON target.match_key = source.match_id
        WHEN MATCHED THEN UPDATE SET
            target.elo_home = source.elo_home,
            target.elo_away = source.elo_away
    """)

In [ ]:
def main():
    """Rate every played match in the gold-scope leagues, oldest first,
    and merge the pre-kick-off ratings onto fact_match.

    Recomputed in full every run. Elo is sequential -- each match depends
    on every match before it -- so there is no incremental shortcut that
    stays correct once history is backfilled or a result is corrected.
    """
    matches = load_matches()
    dim_league = spark.table(DIM_LEAGUE_TABLE)

    scoped = scope_to_gold_leagues(matches, dim_league)
    df_elo = build_elo(scoped)

    merge_into_fact_match(df_elo)
    print(f"Merged Elo ratings for {df_elo.count()} matches into {GOLD_TABLE}")

main()